# Previsão mensal de vendas

Baseline de média móvel de 3 meses para estimar as vendas mensais de **Bússola de Bordo 702**. O treino utiliza dados até dezembro de 2025 e a avaliação considera exclusivamente o primeiro trimestre de 2026.

## 1. Carregando os datasets

In [122]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/1-lh_nautical_csv")

products_df = pd.read_csv(DATA_DIR / "products.csv", parse_dates=["created_at", "updated_at"])
product_variants_df = pd.read_csv(
    DATA_DIR / "product_variants.csv", parse_dates=["created_at", "updated_at"])
orders_df = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["placed_at", "created_at", "updated_at"])
order_items_df = pd.read_csv(DATA_DIR / "order_items.csv")

print(f"products_df: linhas {products_df.shape[0]} colunas {products_df.shape[1]}")
display(products_df.info())
print(f"product_variants_df: linhas {product_variants_df.shape[0]} colunas {product_variants_df.shape[1]}")
display(product_variants_df.info())
print(f"orders_df: linhas {orders_df.shape[0]} colunas {orders_df.shape[1]}")
display(orders_df.info())
print(f"order_items_df: linhas {order_items_df.shape[0]} colunas {order_items_df.shape[1]}")
display(order_items_df.info())

products_df: linhas 500 colunas 10
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id               500 non-null    int64         
 1   name             500 non-null    str           
 2   description      490 non-null    str           
 3   brand_id         500 non-null    int64         
 4   category_id      500 non-null    int64         
 5   ncm_code         500 non-null    int64         
 6   unit_of_measure  500 non-null    str           
 7   is_active        500 non-null    bool          
 8   created_at       500 non-null    datetime64[us]
 9   updated_at       500 non-null    datetime64[us]
dtypes: bool(1), datetime64[us](2), int64(4), str(3)
memory usage: 35.8 KB


None

product_variants_df: linhas 1009 colunas 12
<class 'pandas.DataFrame'>
RangeIndex: 1009 entries, 0 to 1008
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           1009 non-null   int64         
 1   product_id   1009 non-null   int64         
 2   sku          1009 non-null   str           
 3   barcode_ean  852 non-null    float64       
 4   sale_price   1009 non-null   float64       
 5   cost_price   1009 non-null   float64       
 6   weight_kg    1009 non-null   float64       
 7   icms_rate    1009 non-null   float64       
 8   ipi_rate     1009 non-null   float64       
 9   is_active    1009 non-null   bool          
 10  created_at   1009 non-null   datetime64[us]
 11  updated_at   1009 non-null   datetime64[us]
dtypes: bool(1), datetime64[us](2), float64(6), int64(2), str(1)
memory usage: 87.8 KB


None

orders_df: linhas 48998 colunas 13
<class 'pandas.DataFrame'>
RangeIndex: 48998 entries, 0 to 48997
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id               48998 non-null  int64         
 1   order_number     48998 non-null  str           
 2   channel          48998 non-null  str           
 3   customer_id      48998 non-null  int64         
 4   salesperson_id   24867 non-null  float64       
 5   location_id      48998 non-null  int64         
 6   status           48998 non-null  str           
 7   subtotal         48998 non-null  float64       
 8   discount_amount  48998 non-null  float64       
 9   total            48998 non-null  float64       
 10  placed_at        48998 non-null  datetime64[us]
 11  created_at       48998 non-null  datetime64[us]
 12  updated_at       48998 non-null  datetime64[us]
dtypes: datetime64[us](3), float64(4), int64(3), str(3)
memory usage: 4.

None

order_items_df: linhas 147320 colunas 8
<class 'pandas.DataFrame'>
RangeIndex: 147320 entries, 0 to 147319
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  147320 non-null  int64  
 1   order_id            147320 non-null  int64  
 2   product_variant_id  147320 non-null  int64  
 3   quantity            147320 non-null  int64  
 4   unit_price          147320 non-null  float64
 5   icms_rate           147320 non-null  float64
 6   ipi_rate            147320 non-null  float64
 7   line_total          147320 non-null  float64
dtypes: float64(4), int64(4)
memory usage: 9.0 MB


None

Exibindo os df

In [123]:
display(products_df.head(1))
display(product_variants_df.head(1))
display(orders_df.head(1))
display(order_items_df.head(1))

,id,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active,created_at,updated_at
0,1,Motor de Popa 6014,Motor de popa 4 tempos para embarcações de lazer,9,9,50675645,UN,True,2023-06-07 05:32:02,2025-06-05 20:11:50


,id,product_id,sku,barcode_ean,sale_price,cost_price,weight_kg,icms_rate,ipi_rate,is_active,created_at,updated_at
0,1,1,LHN-353137,8.123564e+11,2452.69,1409.59,17.466,18.0,5.0,True,2023-08-31 09:15:26,2026-07-29 01:56:22


,id,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,2022-09-06 05:37:37,2022-09-06 05:37:37


,id,order_id,product_variant_id,quantity,unit_price,icms_rate,ipi_rate,line_total
0,1,1,113,6,53.89,12.0,0.0,323.34


## 2. Unificando os datasets

In [124]:
union_df = (
    orders_df
    .merge(
        order_items_df,
        left_on="id",
        right_on="order_id",
        how="left",
        suffixes=("_order", "_order_item"),
    )
    .merge(
        product_variants_df,
        left_on="product_variant_id",
        right_on="id",
        how="left",
        suffixes=("_order", "_variant"),
    )
    .merge(
        products_df,
        left_on="product_id",
        right_on="id",
        how="left",
        suffixes=("_variant", "_product"),
    )
).copy()

union_df.head(5)

,id_order,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,...,id_product,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active_product,created_at,updated_at
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,...,59,Bateria Náutica 8789,Bateria estacionária 105Ah náutica,11,3,91840820,UN,True,2020-10-25 05:06:59,2026-05-27 05:47:10
1,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,...,146,Cabo Náutico 7323,Cabo de polipropileno trançado,6,2,10813652,M,True,2026-10-30 06:39:37,2026-12-29 21:53:20
2,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,...,180,Motor de Popa 1949,Motor de popa 4 tempos para embarcações de lazer,7,13,22146679,UN,True,2024-11-01 07:14:54,2024-11-10 04:23:05
3,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,...,275,Cabo Náutico 5921,Cabo de polipropileno trançado,2,12,65691283,M,True,2024-10-19 05:21:51,2026-04-01 02:13:41
4,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,...,190,GPS Plotter 3107,GPS chartplotter com cartografia,9,4,95845129,UN,True,2023-06-04 14:31:13,2023-08-28 18:39:31


## 3. Filtrando o produto e as vendas válidas

In [125]:
target_products_df = union_df[
    union_df["name"].eq("Bússola de Bordo 702")
    & union_df["status"].isin(["paid", "confirmed"])
    & union_df["placed_at"].lt("2026-04-01")
    & union_df["is_active_product"].eq(True)
    & union_df["is_active_variant"].eq(True)
].copy()

target_products_df.head()

,id_order,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,...,id_product,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active_product,created_at,updated_at
452,150,SO-000150,pos,1971,6.0,4,paid,32421.38,0.00,32421.38,...,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25
1010,328,SO-000328,ecommerce,1012,9.0,6,paid,13755.20,1513.07,12242.13,...,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25
1208,397,SO-000397,ecommerce,469,6.0,5,paid,46885.00,0.00,46885.00,...,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25
1634,533,SO-000533,ecommerce,1834,NaN,5,confirmed,11658.87,0.00,11658.87,...,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25
1716,563,SO-000563,pos,562,10.0,4,paid,2049.18,0.00,2049.18,...,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25


Criar coluna year-month

In [126]:
target_products_df["year-month"] = (
    target_products_df["placed_at"].dt.to_period("M")
)

display(target_products_df[["placed_at", "year-month"]][
    target_products_df["year-month"].eq("2026-03")
].head(5))

,placed_at,year-month
5855,2026-03-19 00:30:50,2026-03
22037,2026-03-20 17:51:47,2026-03
53348,2026-03-27 20:22:11,2026-03
54090,2026-03-16 17:38:53,2026-03
66675,2026-03-26 16:58:10,2026-03


Agrupando por ano-mês e somando a quantidade vendida total por ano-mes

In [127]:
total_sales_per_month_df = (
    target_products_df
    .groupby("year-month")
    .agg(total_sales=("quantity", "sum"))
    .reset_index()
    [["year-month", "total_sales"]]
)

display(total_sales_per_month_df)

,year-month,total_sales
0,2020-01,29
1,2020-02,16
2,2020-03,17
3,2020-04,28
4,2020-05,5
...,...,...
65,2025-11,54
66,2025-12,19
67,2026-01,76
68,2026-02,55


Separando df de treino e df de teste

In [128]:
train_df = total_sales_per_month_df[
    total_sales_per_month_df["year-month"].le("2025-12")
].copy()

test_df = total_sales_per_month_df[
    total_sales_per_month_df["year-month"].between("2026-01", "2026-03")
].copy()

display(train_df.tail(3))
display(test_df)


,year-month,total_sales
64,2025-10,25
65,2025-11,54
66,2025-12,19


,year-month,total_sales
67,2026-01,76
68,2026-02,55
69,2026-03,51


## 4. Calculando a média móvel

In [131]:
from math import ceil

history = train_df["total_sales"].tolist()

predictions = []

for _ in range(3):
    prediction = ceil(sum(history[-3:]) / 3) # Arredondamento feito para cima, pois não é possível vender uma fração de produto

    predictions.append(prediction)
    history.append(prediction)

test_df["total_sales_predictions"] = predictions

display(test_df)

,year-month,total_sales,total_sales_predictions,absolute_error
67,2026-01,76,33,43
68,2026-02,55,36,19
69,2026-03,51,30,21


Calculando o MAE

In [140]:
test_df["absolute_error"] = abs(test_df["total_sales"] - test_df["total_sales_predictions"])

display(test_df.head(3))
print(f"Mean Absolute Error: {test_df['absolute_error'].mean():.2f}")
print(f"Previsão total de vendas para o primeiro trimestre de 2026: {sum(predictions)}")
print(f"Total de vendas reais no primeiro trimestre de 2026: {sum(test_df['total_sales'].tolist())}")
print(f"Vendas reais - Previsão: {sum(test_df['total_sales'].tolist()) - sum(predictions)}")

,year-month,total_sales,total_sales_predictions,absolute_error
67,2026-01,76,33,43
68,2026-02,55,36,19
69,2026-03,51,30,21


Mean Absolute Error: 27.67
Previsão total de vendas para o primeiro trimestre de 2026: 99
Total de vendas reais no primeiro trimestre de 2026: 182
Vendas reais - Previsão: 83


## Respostas finais

**a. O baseline é adequado para esse produto?**  
Não como modelo final de previsão. O baseline apresentou MAE de 27,67 unidades, ou seja, suas previsões erraram, em média, cerca de 27,67 unidades, o que equivale a 28 unidades por mês. Além disso, em janeiro de 2026 foram previstas 33 unidades, enquanto as vendas reais foram 76, resultando em um erro de 43 unidades. Portanto, ele pode ser utilizado como uma referência simples para comparação com modelos futuros, mas teve baixa precisão no período de teste.

**3. Cite uma limitação desse método.**  
A média móvel considera apenas as vendas recentes e, por isso, não consegue antecipar mudanças bruscas de demanda, tendências ou sazonalidades. Além disso, como sua previsão é recursiva, as previsões de fevereiro e março utilizam previsões anteriores, fazendo com que um erro possa se propagar para os meses seguintes.

**1. Como o baseline foi construído?**
O baseline foi construído utilizando uma média móvel de 3 meses. Para prever janeiro de 2026, foi calculada a média das vendas de outubro, novembro e dezembro de 2025. Para os meses seguintes, a previsão foi feita de forma recursiva, utilizando sempre os três valores mais recentes disponíveis, incluindo previsões anteriores quando necessário.

**2. Como evitou data leakage?**
O data leakage foi evitado separando os dados temporalmente: o conjunto de treino (train_df) contém apenas dados até 31/12/2025, enquanto os dados reais de janeiro a março de 2026 foram mantidos exclusivamente no conjunto de teste (test_df). Durante a geração das previsões, nenhum valor real do período de teste foi utilizado, fevereiro e março foram previstos usando apenas dados do treino e previsões anteriores. Os valores reais do teste só foram usados posteriormente para calcular o MAE.

**Utilizando seu modelo treinado, qual é a soma total da previsão de vendas (arredondada para número inteiro) para o 'Bússola de Bordo 702' durante o primeiro trimestre de 2026?**

99